# Call Center Feature Store
Create feature views from NICE_CALLCENTER_DB.AI_READY tables and generate a training dataset.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

session = get_active_session()

DB = "NICE_CALLCENTER_DB"
FS_SCHEMA = "FEATURE_STORE"
SOURCE_SCHEMA = "AI_READY"
WAREHOUSE = session.get_current_warehouse()

session.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.{FS_SCHEMA}").collect()

fs = FeatureStore(
    session=session,
    database=DB,
    name=FS_SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
print("Feature Store initialized")

In [ ]:
agent_entity = Entity(name="AGENT", join_keys=["AGENT_ID"], desc="Call center agent")
customer_entity = Entity(name="CUSTOMER", join_keys=["CUSTOMER_ID"], desc="Customer")
interaction_entity = Entity(name="INTERACTION", join_keys=["INTERACTION_ID"], desc="Individual interaction")

fs.register_entity(agent_entity)
fs.register_entity(customer_entity)
fs.register_entity(interaction_entity)
print("Entities registered")

In [ ]:
agent_df = session.table(f"{DB}.{SOURCE_SCHEMA}.DIM_AGENT").select(
    F.col("AGENT_ID"),
    F.datediff("day", F.col("HIRE_DATE"), F.current_date()).alias("TENURE_DAYS"),
    F.col("IS_ACTIVE").cast(T.IntegerType()).alias("IS_ACTIVE_FLAG"),
    F.col("PERFORMANCE_TIER_RANK"),
    F.col("AVG_HANDLE_TIME_MIN"),
    F.length(F.col("SKILL_SET")).alias("SKILL_SET_LENGTH"),
    F.iff(F.col("TERMINATION_DATE").is_null(), F.lit(0), F.lit(1)).alias("HAS_TERMINATED")
)

agent_fv = FeatureView(
    name="AGENT_DEMOGRAPHIC_FEATURES",
    entities=[agent_entity],
    feature_df=agent_df,
    refresh_freq="60 minutes",
    desc="Agent demographic and profile features from DIM_AGENT"
)

agent_fv = agent_fv.attach_feature_desc({
    "TENURE_DAYS": "Days since agent hire date",
    "IS_ACTIVE_FLAG": "1 if agent is currently active, 0 otherwise",
    "PERFORMANCE_TIER_RANK": "Agent performance tier rank (1=highest)",
    "AVG_HANDLE_TIME_MIN": "Agent average handle time in minutes",
    "SKILL_SET_LENGTH": "Length of agent skill set string as proxy for skill breadth",
    "HAS_TERMINATED": "1 if agent has a termination date, 0 otherwise"
})
print("Agent demographic feature view created")

In [ ]:
customer_df = session.table(f"{DB}.{SOURCE_SCHEMA}.DIM_CUSTOMER").select(
    F.col("CUSTOMER_ID"),
    F.iff(F.col("ACCOUNT_TYPE") == "Enterprise", F.lit(3),
        F.iff(F.col("ACCOUNT_TYPE") == "Business", F.lit(2), F.lit(1))).alias("ACCOUNT_TYPE_RANK"),
    F.iff(F.col("CONTRACT_TIER") == "Platinum", F.lit(4),
        F.iff(F.col("CONTRACT_TIER") == "Gold", F.lit(3),
            F.iff(F.col("CONTRACT_TIER") == "Silver", F.lit(2), F.lit(1)))).alias("CONTRACT_TIER_RANK"),
    F.col("CONTAINS_PII").cast(T.IntegerType()).alias("CONTAINS_PII_FLAG")
)

customer_fv = FeatureView(
    name="CUSTOMER_PROFILE_FEATURES",
    entities=[customer_entity],
    feature_df=customer_df,
    refresh_freq="60 minutes",
    desc="Customer profile features from DIM_CUSTOMER"
)

customer_fv = customer_fv.attach_feature_desc({
    "ACCOUNT_TYPE_RANK": "Numeric rank of account type (3=Enterprise, 2=Business, 1=Other)",
    "CONTRACT_TIER_RANK": "Numeric rank of contract tier (4=Platinum, 3=Gold, 2=Silver, 1=Other)",
    "CONTAINS_PII_FLAG": "1 if customer record contains PII, 0 otherwise"
})
print("Customer profile feature view created")

In [ ]:
interactions = session.table(f"{DB}.{SOURCE_SCHEMA}.FACT_INTERACTIONS")

agent_interaction_df = interactions.filter(F.col("AGENT_ID").is_not_null()).group_by("AGENT_ID").agg(
    F.count("*").alias("TOTAL_INTERACTIONS"),
    F.avg("DURATION_SECONDS").alias("AVG_DURATION_SEC"),
    F.avg("TOTAL_HANDLE_TIME_SEC").alias("AVG_HANDLE_TIME_SEC"),
    F.avg("WRAP_UP_SECONDS").alias("AVG_WRAP_UP_SEC"),
    F.avg("SENTIMENT_SCORE").alias("AVG_SENTIMENT_SCORE"),
    F.avg("CSAT_SCORE").alias("AVG_CSAT_SCORE"),
    F.avg("NPS_SCORE").alias("AVG_NPS_SCORE"),
    F.avg("CES_SCORE").alias("AVG_CES_SCORE"),
    F.avg(F.col("FIRST_CONTACT_RESOLUTION").cast(T.IntegerType())).alias("FCR_RATE"),
    F.avg(F.col("WAS_TRANSFERRED").cast(T.IntegerType())).alias("TRANSFER_RATE"),
    F.avg(F.col("WAS_ESCALATED").cast(T.IntegerType())).alias("ESCALATION_RATE")
)

agent_interaction_fv = FeatureView(
    name="AGENT_INTERACTION_FEATURES",
    entities=[agent_entity],
    feature_df=agent_interaction_df,
    refresh_freq="60 minutes",
    desc="Aggregated interaction metrics per agent from FACT_INTERACTIONS"
)

agent_interaction_fv = agent_interaction_fv.attach_feature_desc({
    "TOTAL_INTERACTIONS": "Total number of interactions handled by agent",
    "AVG_DURATION_SEC": "Average interaction duration in seconds",
    "AVG_HANDLE_TIME_SEC": "Average total handle time in seconds",
    "AVG_WRAP_UP_SEC": "Average wrap-up time in seconds",
    "AVG_SENTIMENT_SCORE": "Average customer sentiment score",
    "AVG_CSAT_SCORE": "Average customer satisfaction score",
    "AVG_NPS_SCORE": "Average Net Promoter Score",
    "AVG_CES_SCORE": "Average Customer Effort Score",
    "FCR_RATE": "First contact resolution rate",
    "TRANSFER_RATE": "Rate of interactions transferred",
    "ESCALATION_RATE": "Rate of interactions escalated"
})
print("Agent interaction feature view created")

In [ ]:
customer_interaction_df = interactions.filter(F.col("CUSTOMER_ID").is_not_null()).group_by("CUSTOMER_ID").agg(
    F.count("*").alias("CUSTOMER_INTERACTION_COUNT"),
    F.avg("DURATION_SECONDS").alias("CUSTOMER_AVG_DURATION_SEC"),
    F.avg("SENTIMENT_SCORE").alias("CUSTOMER_AVG_SENTIMENT"),
    F.avg("CSAT_SCORE").alias("CUSTOMER_AVG_CSAT"),
    F.avg("NPS_SCORE").alias("CUSTOMER_AVG_NPS"),
    F.avg(F.col("FIRST_CONTACT_RESOLUTION").cast(T.IntegerType())).alias("CUSTOMER_FCR_RATE"),
    F.avg(F.col("WAS_ESCALATED").cast(T.IntegerType())).alias("CUSTOMER_ESCALATION_RATE")
)

customer_interaction_fv = FeatureView(
    name="CUSTOMER_INTERACTION_FEATURES",
    entities=[customer_entity],
    feature_df=customer_interaction_df,
    refresh_freq="60 minutes",
    desc="Aggregated interaction metrics per customer from FACT_INTERACTIONS"
)

customer_interaction_fv = customer_interaction_fv.attach_feature_desc({
    "CUSTOMER_INTERACTION_COUNT": "Total interactions for this customer",
    "CUSTOMER_AVG_DURATION_SEC": "Average interaction duration in seconds",
    "CUSTOMER_AVG_SENTIMENT": "Average sentiment score across interactions",
    "CUSTOMER_AVG_CSAT": "Average CSAT score across interactions",
    "CUSTOMER_AVG_NPS": "Average NPS across interactions",
    "CUSTOMER_FCR_RATE": "First contact resolution rate for customer",
    "CUSTOMER_ESCALATION_RATE": "Escalation rate for customer"
})
print("Customer interaction feature view created")

In [ ]:
qa = session.table(f"{DB}.{SOURCE_SCHEMA}.FACT_QA_EVALUATIONS")

agent_qa_df = qa.filter(F.col("AGENT_ID").is_not_null()).group_by("AGENT_ID").agg(
    F.count("*").alias("TOTAL_EVALUATIONS"),
    F.avg("GREETING_SCORE").alias("AVG_GREETING_SCORE"),
    F.avg("EMPATHY_SCORE").alias("AVG_EMPATHY_SCORE"),
    F.avg("RESOLUTION_SCORE").alias("AVG_RESOLUTION_SCORE"),
    F.avg("COMPLIANCE_SCORE").alias("AVG_COMPLIANCE_SCORE"),
    F.avg("KNOWLEDGE_SCORE").alias("AVG_KNOWLEDGE_SCORE"),
    F.avg("PROFESSIONALISM_SCORE").alias("AVG_PROFESSIONALISM_SCORE"),
    F.avg("OVERALL_SCORE").alias("AVG_OVERALL_QA_SCORE"),
    F.avg("OVERALL_SCORE_PCT").alias("AVG_OVERALL_SCORE_PCT"),
    F.avg(F.col("PASSED").cast(T.IntegerType())).alias("QA_PASS_RATE"),
    F.avg(F.col("COACHING_REQUIRED").cast(T.IntegerType())).alias("COACHING_REQUIRED_RATE")
)

agent_qa_fv = FeatureView(
    name="AGENT_QA_FEATURES",
    entities=[agent_entity],
    feature_df=agent_qa_df,
    refresh_freq="60 minutes",
    desc="Aggregated QA evaluation metrics per agent from FACT_QA_EVALUATIONS"
)

agent_qa_fv = agent_qa_fv.attach_feature_desc({
    "TOTAL_EVALUATIONS": "Total QA evaluations for agent",
    "AVG_GREETING_SCORE": "Average greeting score",
    "AVG_EMPATHY_SCORE": "Average empathy score",
    "AVG_RESOLUTION_SCORE": "Average resolution score",
    "AVG_COMPLIANCE_SCORE": "Average compliance score",
    "AVG_KNOWLEDGE_SCORE": "Average knowledge score",
    "AVG_PROFESSIONALISM_SCORE": "Average professionalism score",
    "AVG_OVERALL_QA_SCORE": "Average overall QA score",
    "AVG_OVERALL_SCORE_PCT": "Average overall score percentage",
    "QA_PASS_RATE": "Rate of QA evaluations passed",
    "COACHING_REQUIRED_RATE": "Rate of evaluations requiring coaching"
})
print("Agent QA feature view created")

In [ ]:
interaction_detail_df = interactions.filter(
    F.col("INTERACTION_ID").is_not_null()
).select(
    F.col("INTERACTION_ID"),
    F.col("DURATION_SECONDS"),
    F.col("DURATION_MINUTES"),
    F.col("WRAP_UP_SECONDS"),
    F.col("TOTAL_HANDLE_TIME_SEC"),
    F.col("WAS_TRANSFERRED").cast(T.IntegerType()).alias("WAS_TRANSFERRED_FLAG"),
    F.col("WAS_ESCALATED").cast(T.IntegerType()).alias("WAS_ESCALATED_FLAG"),
    F.col("SENTIMENT_SCORE"),
    F.col("CSAT_SCORE"),
    F.col("NPS_SCORE"),
    F.col("CES_SCORE"),
    F.col("FIRST_CONTACT_RESOLUTION").cast(T.IntegerType()).alias("FCR_FLAG")
)

interaction_fv = FeatureView(
    name="INTERACTION_DETAIL_FEATURES",
    entities=[interaction_entity],
    feature_df=interaction_detail_df,
    refresh_freq="60 minutes",
    desc="Per-interaction detail features from FACT_INTERACTIONS"
)
print("Interaction detail feature view created")

In [ ]:
reg_agent_fv = fs.register_feature_view(feature_view=agent_fv, version="v1", block=True, overwrite=True)
print("Registered: AGENT_DEMOGRAPHIC_FEATURES v1")

reg_customer_fv = fs.register_feature_view(feature_view=customer_fv, version="v1", block=True, overwrite=True)
print("Registered: CUSTOMER_PROFILE_FEATURES v1")

reg_agent_interaction_fv = fs.register_feature_view(feature_view=agent_interaction_fv, version="v1", block=True, overwrite=True)
print("Registered: AGENT_INTERACTION_FEATURES v1")

reg_customer_interaction_fv = fs.register_feature_view(feature_view=customer_interaction_fv, version="v1", block=True, overwrite=True)
print("Registered: CUSTOMER_INTERACTION_FEATURES v1")

reg_agent_qa_fv = fs.register_feature_view(feature_view=agent_qa_fv, version="v1", block=True, overwrite=True)
print("Registered: AGENT_QA_FEATURES v1")

reg_interaction_fv = fs.register_feature_view(feature_view=interaction_fv, version="v1", block=True, overwrite=True)
print("Registered: INTERACTION_DETAIL_FEATURES v1")

print("\nAll feature views registered successfully!")

In [ ]:
fs.list_feature_views().show()

In [ ]:
spine_df = session.table(f"{DB}.{SOURCE_SCHEMA}.FACT_INTERACTIONS").filter(
    (F.col("AGENT_ID").is_not_null()) & (F.col("CUSTOMER_ID").is_not_null())
).select(
    F.col("INTERACTION_ID"),
    F.col("AGENT_ID"),
    F.col("CUSTOMER_ID"),
    F.col("CSAT_SCORE").alias("LABEL_CSAT_SCORE"),
    F.col("FIRST_CONTACT_RESOLUTION").cast(T.IntegerType()).alias("LABEL_FCR")
)

training_set = fs.generate_training_set(
    spine_df=spine_df,
    features=[
        reg_agent_fv,
        reg_customer_fv,
        reg_agent_interaction_fv,
        reg_customer_interaction_fv,
        reg_agent_qa_fv,
        reg_interaction_fv
    ],
    spine_label_cols=["LABEL_CSAT_SCORE", "LABEL_FCR"],
    save_as="CALLCENTER_TRAINING_DATA",
)

print("Training dataset generated and saved as CALLCENTER_TRAINING_DATA")
print(f"Shape: {training_set.count()} rows")
training_set.show(5)